## Imports

In [1]:
import math
import os
import time
import requests
import pandas as pd
import json
import numpy as np
from pathlib import Path
from typing import Optional, Dict, List, Union
from unidecode import unidecode
import matplotlib.pyplot as plt

import pandas_gbq
from google.auth import default
from google.cloud import bigquery
from google.api_core.exceptions import NotFound

import warnings
warnings.filterwarnings("ignore")

/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/google/api_core/_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.9.6). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please u

In [2]:
from funcoes_escoragem import *

## Diretório

In [3]:
BASE_DIR = Path("data")
RAW_DIR = BASE_DIR / "raw"
TRUSTED_DIR = BASE_DIR / "trusted"
ANALYTICS_DIR = BASE_DIR / "analytics"

for path in [RAW_DIR, TRUSTED_DIR, ANALYTICS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

## Google BigQuery

In [4]:
project_id = 'loft-dl-fintech'

# CredPago - Consulta Realizada**

In [5]:
query_credpago = f"""
WITH consulta_realizada AS (
    SELECT
        CAST(REGEXP_REPLACE(documento, r'[^0-9]', '') AS INT64) AS CPF_CNPJ,
        id_externo AS contract_id,

        MIN(DATE(data)) OVER (
            PARTITION BY id_externo
        ) AS requested_at,

        MAX(DATE(data)) OVER (
            PARTITION BY id_externo
        ) AS data_ultima_consulta,
        ARRAY_LENGTH(JSON_EXTRACT_ARRAY(CR.json_retornado, '$.pessoas')) AS qtd_proponentes,
        CR.*
    FROM `loft-dl-fintech.bronze_credpago_sortinghat.consulta_realizada` CR
    WHERE DATE(data) >= DATE_SUB(CURRENT_DATE(), INTERVAL 12 WEEK)
    AND DATE(data) < CURRENT_DATE()
)

SELECT *
FROM consulta_realizada
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY contract_id
    ORDER BY 
        CASE WHEN model = 'BLEND_4' THEN 1 ELSE 2 END ASC,
        data DESC
) = 1
"""

credpago_df = pd.read_gbq(query_credpago, project_id=project_id)
credpago_df

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,renda_considerada,model,id_externo,json_retornado,...,id_funcionalidade,_sdc_table_version,request,_sdc_received_at,_sdc_sequence,documento,rating,_sdc_batched_at,data,result
0,5171812071,4098616,2026-05-06,2026-05-06,1,D,0E-9,BVS_CUSTOM,4098616,"{""manual"":false,""node"":null,""pessoas"":[{""nome""...",...,28,1778785248777,None,2026-05-20 08:01:24+00:00,1779264082305891179,05171812071,E,2026-05-20 08:09:22.484000+00:00,2026-05-06 10:47:07+00:00,REPROVAR
1,2439625175,4192798,2026-05-29,2026-05-29,1,B,0E-9,BLEND3_3,4192798,"{""pessoas"":[{""nome"":"""",""documento"":""0243962517...",...,28,1778785248777,"{""valor_aluguel"":""2420"",""imobiliaria_id"":3366,...",2026-05-29 18:00:33+00:00,1780077632294650448,02439625175,A,2026-05-29 18:09:39.229000+00:00,2026-05-29 14:33:18+00:00,APROVAR
2,10485562987,4268294,2026-06-18,2026-06-18,1,B,0E-9,BLEND3_3,4268294,"{""pessoas"":[{""nome"":""CPF NAO CONFIRMADO NO CAD...",...,28,1778785248777,"{""valor_aluguel"":""1200"",""imobiliaria_id"":16986...",2026-06-18 18:00:35+00:00,1781805630434199919,10485562987,E,2026-06-18 18:10:17.998000+00:00,2026-06-18 10:32:43+00:00,REPROVAR
3,17145554615,4315575,2026-06-30,2026-06-30,1,A,0E-9,BLEND3_3,4315575,"{""pessoas"":[{""nome"":"""",""documento"":""1714555461...",...,28,1778785248777,"{""valor_aluguel"":1600,""matchmaking_on"":false,""...",2026-07-01 08:00:35+00:00,1782892833277155334,17145554615,B,2026-07-01 08:05:22.777000+00:00,2026-06-30 17:19:38+00:00,APROVAR
4,11970207213,4267546,2026-06-18,2026-06-18,2,NI,0E-9,HVA3,4267546,"{""pessoas"":[{""nome"":""CPF NAO CONFIRMADO NO CAD...",...,28,1778785248777,"{""valor_aluguel"":""700"",""imobiliaria_id"":17863,...",2026-06-18 18:00:35+00:00,1781805629240669148,11970207213,E,2026-06-18 18:10:17.756000+00:00,2026-06-18 09:09:06+00:00,REPROVAR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
327931,3501497121,4401542,2026-07-21,2026-07-21,1,NI,3493.500000000,BLEND3_3,4401542,"{""pessoas"":[{""nome"":""LUCAS LEANDRO WALL BRUNO""...",...,31,1778785248777,"{""valor_aluguel"":2950,""matchmaking_on"":false,""...",2026-07-22 08:00:27+00:00,1784707223168745620,03501497121,A,2026-07-22 08:09:19.682000+00:00,2026-07-21 15:18:29+00:00,APROVAR
327932,5794834102,4402821,2026-07-21,2026-07-21,1,B,6096.500000000,BLEND3_3,4402821,"{""pessoas"":[{""nome"":""JORGE BRUNO LARES DE CARV...",...,31,1778785248777,"{""valor_aluguel"":2000,""matchmaking_on"":false,""...",2026-07-22 08:00:27+00:00,1784707225071191792,05794834102,A,2026-07-22 08:09:20.209000+00:00,2026-07-21 17:12:07+00:00,APROVAR
327933,40786812800,4401600,2026-07-21,2026-07-21,1,NI,2397.500000000,BLEND_4,4401600,"{""pessoas"":[{""nome"":""JOYCE NASCIMENTO DE BESSA...",...,31,1778785248777,"{""principalDocument"":""40786812800"",""imobiliari...",2026-07-22 08:00:27+00:00,1784707223341879194,40786812800,A,2026-07-22 08:09:19.706000+00:00,2026-07-21 15:23:54+00:00,APROVAR
327934,1265227071,4402750,2026-07-21,2026-07-21,1,D,13426.000000000,BLEND3_3,4402750,"{""pessoas"":[{""nome"":""LUCIANO DE MORAES"",""docum...",...,32,1778785248777,"{""valor_aluguel"":4500,""matchmaking_on"":false,""...",2026-07-22 08:00:27+00:00,1784707224943523211,01265227071,B,2026-07-22 08:09:20.177000+00:00,2026-07-21 17:04:08+00:00,APROVAR


In [6]:
mask = credpago_df['contract_id'].astype(str).str.match(r'^\d+$')
credpago_df = credpago_df[mask].copy()
credpago_df['contract_id'] = credpago_df['contract_id'].astype(int)

## Quebrar JSON - Retornado

In [7]:
def unwrap_payload(obj):
    """Supports old format (message wrapper) and new format (root payload)."""
    if not obj:
        return None
    if isinstance(obj, dict) and "message" in obj:
        return obj["message"]
    return obj

def get_pessoas(obj):
    payload = unwrap_payload(obj)
    return (payload or {}).get("pessoas") or []

In [8]:
parsed = credpago_df["json_retornado"].apply(
    lambda x: json.loads(x) if pd.notna(x) and x else None
)

valid_parsed = parsed.dropna()
payload_parsed = valid_parsed.apply(unwrap_payload)

# json_normalize resets the index; preserve credpago_df labels for the join below
contrato_df = pd.json_normalize(payload_parsed.tolist(), sep="_")
contrato_df.index = payload_parsed.index
contrato_df = contrato_df.add_prefix("message_")  # keeps message_decisao, message_scores_*, etc.

pessoa_records = payload_parsed.apply(lambda x: (get_pessoas(x) or [{}])[0])
pessoa_df = pd.json_normalize(pessoa_records.tolist(), sep="_")
pessoa_df.index = payload_parsed.index
pessoa_df = pessoa_df.add_prefix("pessoa_")

In [9]:
valid = parsed.notna()
base_idx = credpago_df.loc[valid].index

pendencias = []
for idx, row in parsed[valid].items():
    for p in get_pessoas(row):
        if not isinstance(p, dict):
            continue

        serasa = p.get("anotacoesFinanceirasSerasa") or {}
        pend_list = serasa.get("PendenciasPagamentoPF") or []

        for pend in pend_list:
            if isinstance(pend, dict):
                pendencias.append({"idx": idx, **pend})

pendencias_df = pd.DataFrame(pendencias)

if not pendencias_df.empty:
    pendencias_agg = (
        pendencias_df
        .groupby("idx", as_index=False)
        .agg(
            qtde_pefin=("Valor", "count"),
            valor_pefin_total=("Valor", lambda s: pd.to_numeric(s, errors="coerce").sum()),
            modalidades_pefin=("Modalidade", lambda s: ",".join(sorted(set(s.dropna().astype(str))))),
        )
    )
else:
    pendencias_agg = pd.DataFrame(columns=["idx", "qtde_pefin", "valor_pefin_total", "modalidades_pefin"])

## Quebrar JSON - Request


In [10]:
request_parsed = credpago_df["request"].apply(
    lambda x: json.loads(x) if pd.notna(x) and x else None
)

request_valid = request_parsed.dropna()
request_df = pd.json_normalize(request_valid.tolist(), sep="_")
request_df.index = request_valid.index
request_df = request_df.add_prefix("request_")

# Unify old (snake_case) and new (camelCase) request schemas
REQUEST_FIELD_ALIASES = {
    "request_valorAluguel": ["request_valor_aluguel"],
    "request_imobiliariaId": ["request_imobiliaria_id"],
    "request_idExterno": ["request_id_externo"],
    "request_imovelTipo": ["request_imovel_tipo"],
    "request_principalDocument": ["request_pessoa_principal_documento"],
}

for target, sources in REQUEST_FIELD_ALIASES.items():
    existing_sources = [c for c in sources if c in request_df.columns]
    if not existing_sources and target not in request_df.columns:
        continue
    if target not in request_df.columns:
        request_df[target] = pd.NA
    for source in existing_sources:
        if target == "request_valorAluguel":
            request_df[target] = request_df[target].combine_first(
                pd.to_numeric(request_df[source], errors="coerce")
            )
        else:
            request_df[target] = request_df[target].combine_first(request_df[source])
        request_df = request_df.drop(columns=[source])


## Join Json's

In [11]:
EXPANDED_PREFIXES = ("message_", "pessoa_", "request_")
expanded_cols = [c for c in credpago_df.columns if c.startswith(EXPANDED_PREFIXES)]

base_df = credpago_df.loc[valid].drop(columns=expanded_cols, errors="ignore")

credpago_expandido = (
    base_df
    .join(contrato_df, how="left")
    .join(pessoa_df, how="left")
    .join(request_df, how="left")
    .reset_index(names="idx")
    .merge(pendencias_agg, on="idx", how="left")
    .drop(columns="idx")
)


In [12]:
cols_drop = [
    # ingestão
    "_sdc_table_version", "_sdc_received_at", "_sdc_sequence", "_sdc_batched_at",

    # json bruto / containers
    "json_retornado",
    "request",
    "message_pessoas", "message_scores", "message_ratings",
    "pessoa_scores", "pessoa_ratings", "message_blend3Variables",
    "request_priorDecisions", "request_dadosAusentes", "request_errosTecnicos",
    "request_outras_pessoas", "request_blend3Variables", "request_scores",

    # bronze redundante
    "score_bvs",
    "renda_considerada",
    "decisao_bureau",
    "limite_mensal",

    # prováveis duplicatas
    "id_externo",
    "data",
    "request_month",

    # operacional / baixo valor (old format only — safe to keep)
    "success", "message_manual", "message_node",
    "message_reaproveitamentoConsultaMotor", "message_savingBureauPowerCurve",
    "message_logs", "message_partnerIds",
    "message_errosTecnicos", "message_dadosAusentes",
    "pessoa_errosTecnicos", "pessoa_dadosAusentes",
    "message_rentGuaranteeConstraints_rent_coverage_multiplier",
    "message_rentGuaranteeConstraints_exit_cost_multiplier",
    "message_rentGuaranteeConstraints_commission_percent",
    "message_rentGuaranteeConstraints_version",
]

credpago_clean = credpago_expandido.drop(columns=[c for c in cols_drop if c in credpago_expandido.columns])


In [13]:
credpago_clean[
    (pd.to_datetime(credpago_clean["requested_at"]).dt.normalize() == "2026-07-03")
    & (credpago_clean["message_decisao"] == "BLEND_4")
][["message_blend3Variables_realEstatePc4HistoryOver100Contracts", "message_blend3Variables_cityPc4HistoryOver100Contracts"]]

,message_blend3Variables_realEstatePc4HistoryOver100Contracts,message_blend3Variables_cityPc4HistoryOver100Contracts
345,0.000000,0.117021
570,0.000000,0.017544
1046,0.000000,0.055249
1115,0.000000,0.074450
3181,0.000000,0.072148
...,...,...
321235,0.000000,0.120879
321286,0.000000,0.076923
321582,0.133065,0.123300
321872,NaN,0.058575


## Escoragem Blend4

In [14]:
credpago_clean_rep = credpago_clean.replace({
    "request_imovelTipo": {"RESIDENCIAL": 1, "COMERCIAL": 0}, 
    "message_blend3Variables_hasPreviousContracts": {True: 1, False: 0},
    "message_blend3Variables_hadOverdueInvoiceInPreviousContracts": {True: 1, False: 0}
}
)

In [15]:
rename_dict = {
    "message_scores_BVS_CUSTOM_V2": "score_proposto__bvs",#
    "message_scores_HFT1": "SERASA_CHSV5",
    "pessoa_idade": "age",
    "request_imovelTipo": "property_type",
    "message_qtdeRestricoesContrato": "qtde_restricoes__consulta_realizada",
    "request_valorAluguel": "rental_value",
    "message_rendaConsideradaContrato": "income",
    "message_blend3Variables_realEstatePc4HistoryOver100Contracts": "agency_pc4_mais_100_contratos__pc_categorias",
    "message_blend3Variables_cityPc4HistoryOver100Contracts": "city_pc4_mais_100_contratos__pc_categorias",
    "message_blend3Variables_hasPreviousContracts": "flag_tem__contratos_anteriores",
    "message_blend3Variables_hadOverdueInvoiceInPreviousContracts": "flag_teve_boleto_atrasado__contratos_anteriores",
}

In [16]:
vars_blend4 = ['score_proposto__bvs', 'SERASA_CHSV5', 'age', 'property_type', 'qtde_restricoes__consulta_realizada', 'rental_value', 'income', 'agency_pc4_mais_100_contratos__pc_categorias', 'city_pc4_mais_100_contratos__pc_categorias', 'flag_tem__contratos_anteriores', 'flag_teve_boleto_atrasado__contratos_anteriores']

id_vars = ['contract_id', 'requested_at', 'CPF_CNPJ', 'message_decisao', 'message_blendRegressaoPredict']

In [17]:
df_predict = (credpago_clean_rep.copy()).rename(columns=rename_dict)
df_predict["REGRA_BLEND_4"] = np.where(
    df_predict["score_proposto__bvs"] <= 334,
    "E_BVS",
    "BLEND4",
)

df_predict["SCORE_BVS"] = df_predict["score_proposto__bvs"]
df_predict["SCORE_SERASA"] = df_predict["SERASA_CHSV5"]
df_predict["RENDA"] = df_predict["income"]

df_predict.head()

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,model,id_consulta,id_funcionalidade,documento,...,request_blend3Variables_cityPc4HistoryOver100Contracts,request_blend3Variables_realEstatePc4HistoryOver100Contracts,request_scores_HFT1,qtde_pefin,valor_pefin_total,modalidades_pefin,REGRA_BLEND_4,SCORE_BVS,SCORE_SERASA,RENDA
0,5171812071,4098616,2026-05-06,2026-05-06,1,D,BVS_CUSTOM,5755998,28,05171812071,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,0.0
1,2439625175,4192798,2026-05-29,2026-05-29,1,B,BLEND3_3,5873519,28,02439625175,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,0.0
2,10485562987,4268294,2026-06-18,2026-06-18,1,B,BLEND3_3,5966811,28,10485562987,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,0.0
3,17145554615,4315575,2026-06-30,2026-06-30,1,A,BLEND3_3,6025449,28,17145554615,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,0.0
4,11970207213,4267546,2026-06-18,2026-06-18,2,NI,HVA3,5965891,28,11970207213,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,0.0


In [18]:
df_predict.groupby("REGRA_BLEND_4", dropna=False).size()

REGRA_BLEND_4
BLEND4    326916
E_BVS       1019
dtype: int64

In [19]:
# df_predict.to_csv(ANALYTICS_DIR/"df_predict_blend4_pre.csv", index=False)

## Criacão das Variáveis

In [20]:
df_predict_vars = prepare_blend4_variables(df_predict)
df_predict_escorada = predict_blend4_1(df_predict_vars)

## Criar Rating Blend4

In [21]:
score = pd.to_numeric(df_predict_escorada["pred_blend4_1_to_score"], errors="coerce")

conditions = [
    score.between(480, 1000),
    score.between(443, 479),
    score.between(408, 442),
    score.between(343, 407),
    score.between(0, 342),
]

choices = ["A", "B", "C", "D", "E"]

df_predict_escorada["rating_manual_blend4"] = np.select(conditions, choices, default=None)

In [22]:
score = pd.to_numeric(df_predict_escorada["message_blendRegressaoPredict"], errors="coerce")

conditions = [
    score.between(480, 1000),
    score.between(443, 479),
    score.between(408, 442),
    score.between(343, 407),
    score.between(0, 342),
]

choices = ["A", "B", "C", "D", "E"]

df_predict_escorada["rating_json_blend4"] = np.select(conditions, choices, default=None)

## Salvar Base como se fosse Blend4

In [23]:
# df_predict_blend4 = df_predict_escorada[df_predict_escorada["message_decisao"] == "BLEND3_3"].copy()
# df_predict_blend4["message_decisao"] = df_predict_blend4["message_decisao"].replace("BLEND3_3", "BLEND4")

# cp_final_df = pd.concat([df_predict_escorada, df_predict_blend4])
cp_final_df = df_predict_escorada.copy()
cp_final_df

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,model,id_consulta,id_funcionalidade,documento,...,SERASA_CHSV5__normalized4_1,age__normalized4_1,qtde_restricoes__consulta_realizada__normalized4_1,income_commitment__normalized4_1,agency_pc4_mais_100_contratos__pc_categorias__normalized4_1,city_pc4_mais_100_contratos__pc_categorias__normalized4_1,pred_blend4_1,pred_blend4_1_to_score,rating_manual_blend4,rating_json_blend4
0,5171812071,4098616,2026-05-06,2026-05-06,1,D,BVS_CUSTOM,5755998,28,05171812071,...,0.000000,-1.70,0.0,0.000000,0.000000,0.000000,0.492096,508.0,A,None
1,2439625175,4192798,2026-05-29,2026-05-29,1,B,BLEND3_3,5873519,28,02439625175,...,0.000000,0.00,0.0,0.000000,-0.091927,-0.985010,0.510464,490.0,A,None
2,10485562987,4268294,2026-06-18,2026-06-18,1,B,BLEND3_3,5966811,28,10485562987,...,0.000000,-1.70,0.0,0.000000,-0.020166,0.106385,0.550732,449.0,B,D
3,17145554615,4315575,2026-06-30,2026-06-30,1,A,BLEND3_3,6025449,28,17145554615,...,0.000000,0.00,0.0,0.000000,0.000000,-1.459541,0.460033,540.0,A,None
4,11970207213,4267546,2026-06-18,2026-06-18,2,NI,HVA3,5965891,28,11970207213,...,0.000000,-1.70,0.0,0.000000,0.000000,0.000000,0.492096,508.0,A,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
327930,3501497121,4401542,2026-07-21,2026-07-21,1,NI,BLEND3_3,6131946,31,03501497121,...,0.000000,-0.55,0.0,0.915545,0.000000,-1.137519,0.492979,507.0,A,A
327931,5794834102,4402821,2026-07-21,2026-07-21,1,B,BLEND3_3,6133561,31,05794834102,...,0.000000,-0.50,0.0,-0.225059,0.099036,1.883000,0.606706,393.0,D,A
327932,40786812800,4401600,2026-07-21,2026-07-21,1,NI,BLEND_4,6132021,31,40786812800,...,1.000000,-0.20,0.0,0.708696,0.000000,0.000000,0.262795,737.0,A,A
327933,1265227071,4402750,2026-07-21,2026-07-21,1,D,BLEND3_3,6133463,32,01265227071,...,0.000000,0.20,0.0,-0.209347,0.000000,1.027005,0.541975,458.0,B,C


In [24]:
cp_final_df.groupby("message_decisao", dropna=False).size()

message_decisao
                            41
BLEND3_3                278001
BLEND_4                   9476
BLEND_REGRESSAO_2026     21123
BVS_CUSTOM                4282
BVS_CUSTOM_V2              155
HFT1                       123
HVA3                     12777
HVA4                      1957
dtype: int64

In [25]:
bvs = pd.to_numeric(cp_final_df["SCORE_BVS"], errors="coerce")
score = pd.to_numeric(cp_final_df["pred_blend4_1_to_score"], errors="coerce")

conditions = [
    bvs <= 334,                         # corte customizado BVS → E
    score.between(763, 1000),           # 763 – 1000
    score.between(704, 762),            # 704 – 762
    score.between(653, 703),            # 653 – 703
    score.between(607, 652),            # 607 – 652
    score.between(562, 606),            # 562 – 606
    score.between(520, 561),            # 520 – 561
    score.between(480, 519),            # 480 – 519
    score.between(443, 479),            # 443 – 479
    score.between(408, 442),            # 408 – 442
    score.between(375, 407),            # 375 – 407
    score.between(343, 374),            # 343 – 374
    score.between(307, 342),            # 307 – 342
    score.between(0, 306),              # 0 – 306
]

choices = [
    "E.E",      # override BVS ≤ 334
    "1.A+",
    "2.A",
    "3.A",
    "4.B+",
    "5.B+",
    "6.B",
    "7.B",
    "8.C",
    "9.D+",
    "10.D",
    "11.D",
    "E-1.E",
    "E-2.E",
]

cp_final_df["rating_cl_pol_blend4"] = np.select(conditions, choices, default=None)

## Salvar

In [26]:
cp_final_df.groupby("REGRA_BLEND_4", dropna=False).size()

REGRA_BLEND_4
BLEND4    326916
E_BVS       1019
dtype: int64

In [27]:
cp_final_df.groupby("message_decisao", dropna=False).size()

message_decisao
                            41
BLEND3_3                278001
BLEND_4                   9476
BLEND_REGRESSAO_2026     21123
BVS_CUSTOM                4282
BVS_CUSTOM_V2              155
HFT1                       123
HVA3                     12777
HVA4                      1957
dtype: int64

In [28]:
cp_final_df.groupby("model", dropna=False).size()

model
                            39
BLEND3_3                278001
BLEND_4                   9476
BLEND_REGRESSAO_2026     21123
BVS_CUSTOM                4282
BVS_CUSTOM_V2              155
HFT1                       123
HVA3                     12777
HVA4                      1957
NaN                          2
dtype: int64

In [29]:
cp_final_df.groupby("REGRA_BLEND_4", dropna=False).size()

REGRA_BLEND_4
BLEND4    326916
E_BVS       1019
dtype: int64

In [30]:
cp_final_df.to_csv(ANALYTICS_DIR/"df_predict_blend4.csv", index=False)